# Data Cleaning

## Overview

Data cleaning is the process of detecting and correcting structural problems — wrong types, inconsistent categories, duplicates, out-of-range values, and encoding errors — before any analysis. Cleaning is distinct from imputation (handled in `missing_data.ipynb`).

**Cleaning checklist:**

| Problem | Detection | Fix |
|---|---|---|
| Wrong dtype | `.dtypes`, `.info()` | `.astype()`, `pd.to_numeric()` |
| Inconsistent categories | `.value_counts()` | `.str.strip()`, `.replace()`, `.map()` |
| Duplicates | `.duplicated()` | `.drop_duplicates()` |
| Out-of-range values | `.describe()`, plots | Clip, flag, or remove |
| Column name issues | `.columns` | `.rename()`, `.str.lower()` |

---

In [13]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# Simulate a messy dataset — typical field data entry problems
n = 150
raw = pd.DataFrame({
    'Site ID ':      [f'SITE_{i:03d}' for i in range(1, n+1)],   # trailing space in name
    'Catchment':     rng.choice(['North', 'north', ' South', 'East', 'EAST', 'West'], n),
    'Elevation(m)':  np.where(rng.random(n) < 0.05, -999, rng.uniform(50, 400, n)).round(1),
    'nitrate':       rng.gamma(2, 2, n).round(2),
    'Nitrate ':      rng.gamma(2, 2, n).round(2),   # duplicate with different name
    'richness':      rng.integers(5, 35, n).astype(str),   # numeric stored as string
    'treatment':     rng.choice(['control', 'Control', 'restored', 'Restored', ''], n),
    'sample_date':   rng.choice(['2023-06-01', '2023-06-15', '01/07/2023', '15 Jul 2023'], n),
})

# Introduce duplicates
raw = pd.concat([raw, raw.iloc[[0, 1, 2]]], ignore_index=True)

print(f'Raw shape: {raw.shape}')
raw.head()

Raw shape: (153, 8)


,Site ID,Catchment,Elevation(m),nitrate,Nitrate,richness,treatment,sample_date
0,SITE_001,North,373.7,4.42,1.56,29,restored,15 Jul 2023
1,SITE_002,EAST,58.7,1.28,7.55,11,control,15 Jul 2023
2,SITE_003,East,244.3,8.81,0.60,11,restored,15 Jul 2023
3,SITE_004,South,271.9,1.63,1.50,32,restored,2023-06-15
4,SITE_005,South,87.1,6.02,3.13,20,Restored,2023-06-15


---
## Step 1: Standardise Column Names

In [14]:
# Standardise: lowercase, strip spaces, replace special chars
df = raw.copy()
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r'[^a-z0-9_]', '_', regex=True)
    .str.replace(r'_+', '_', regex=True)
    .str.strip('_')
)
print('Cleaned columns:', df.columns.tolist())

# Drop exact duplicate columns (same data, different name)
df = df.loc[:, ~df.columns.duplicated()]
# Robustly normalise a few common names that may vary after cleaning
# e.g. 'Elevation(m)' -> 'elevation_m' -> we want 'elevation'
# and 'Site ID ' -> 'site_id' (may already be correct)
# Find any column containing 'elevation' and rename to 'elevation'
elev_cols = [c for c in df.columns if 'elevation' in c]
if elev_cols:
    # keep the first matching column name as 'elevation'
    df = df.rename(columns={elev_cols[0]: 'elevation'})
# Find site id-like columns and normalise to 'site_id'
site_cols = [c for c in df.columns if c.startswith('site_id') or 'site id' in c]
if site_cols:
    df = df.rename(columns={site_cols[0]: 'site_id'})
print('After dedup / rename:', df.columns.tolist())

Cleaned columns: ['site_id', 'catchment', 'elevation_m', 'nitrate', 'nitrate', 'richness', 'treatment', 'sample_date']
After dedup / rename: ['site_id', 'catchment', 'elevation', 'nitrate', 'richness', 'treatment', 'sample_date']


---
## Step 2: Remove Duplicate Rows

In [15]:
print(f'Duplicated rows: {df.duplicated().sum()}')

# Exact duplicates
df = df.drop_duplicates()

# Duplicate on key column only (same site_id, different other values = data entry error)
key_dupes = df.duplicated(subset=['site_id'], keep=False)
print(f'Duplicate site_ids: {key_dupes.sum()}')
# Keep first occurrence; flag ambiguous cases for review in practice
df = df.drop_duplicates(subset=['site_id'], keep='first')
print(f'Shape after dedup: {df.shape}')

Duplicated rows: 3
Duplicate site_ids: 0
Shape after dedup: (150, 7)


---
## Step 3: Fix Data Types

In [17]:
# richness stored as string — convert; errors='coerce' turns unparseable to NaN
df['richness'] = pd.to_numeric(df['richness'], errors='coerce')

# Elevation: -999 is a sentinel for missing — replace with NaN
df['elevation'] = df['elevation'].replace(-999, np.nan)

# Sample date: mixed formats — parse to datetime (drop deprecated arg)
df['sample_date'] = pd.to_datetime(df['sample_date'], errors='coerce')
print('Unparseable dates:', df['sample_date'].isnull().sum())

print(df.dtypes)

Unparseable dates: 116
site_id                   str
catchment                 str
elevation             float64
nitrate               float64
richness                int64
treatment                 str
sample_date    datetime64[us]
dtype: object


---
## Step 4: Standardise Categorical Values

In [18]:
# Find the actual column name for catchment (robust to small variations)
catch_cols = [c for c in df.columns if 'catchment' in c]
if catch_cols:
    catch_col = catch_cols[0]
    print('Raw catchment values:', df[catch_col].value_counts().to_dict())
    # Normalise case and whitespace first
    df[catch_col] = df[catch_col].astype(str).str.strip().str.title()
    print('Cleaned catchment values:', df[catch_col].value_counts().to_dict())
    # Cast to pandas Categorical for memory efficiency
    df[catch_col] = pd.Categorical(df[catch_col])
    # Also ensure a standard column name 'catchment'
    if catch_col != 'catchment':
        df = df.rename(columns={catch_col: 'catchment'})
else:
    print('Warning: no catchment-like column found; skipping catchment cleaning')

# Treatment: handle missing/empty strings and standardise
treat_cols = [c for c in df.columns if 'treatment' in c]
if treat_cols:
    treat_col = treat_cols[0]
    df[treat_col] = df[treat_col].astype(str).str.strip().replace({'': np.nan})
    df[treat_col] = df[treat_col].str.lower()
    print('Treatment values:', df[treat_col].value_counts(dropna=False).to_dict())
    df[treat_col] = pd.Categorical(df[treat_col], categories=['control', 'restored'])
    if treat_col != 'treatment':
        df = df.rename(columns={treat_col: 'treatment'})
else:
    print('Warning: no treatment-like column found; skipping treatment cleaning')

Raw catchment values: {'EAST': 37, ' South': 28, 'North': 24, 'East': 24, 'north': 19, 'West': 18}
Cleaned catchment values: {'East': 61, 'North': 43, 'South': 28, 'West': 18}
Treatment values: {'restored': 60, 'control': 60, nan: 30}


---
## Step 5: Flag Out-of-Range Values

In [19]:
# Define valid ranges from domain knowledge
valid_ranges = {
    'elevation': (0, 500),
    'nitrate':   (0, 50),
    'richness':  (0, 60),
}

# Flag rather than silently drop — document what was removed
flags = pd.DataFrame({'site_id': df['site_id']})
for col, (lo, hi) in valid_ranges.items():
    out_of_range = ~df[col].between(lo, hi, inclusive='both') & df[col].notna()
    flags[f'{col}_flag'] = out_of_range
    if out_of_range.any():
        print(f'{col}: {out_of_range.sum()} out-of-range values')
        print(df.loc[out_of_range, col].describe())

print(f'\nClean dataset shape: {df.shape}')
print(f'Missing values:\n{df.isnull().sum()}')


Clean dataset shape: (150, 7)
Missing values:
site_id          0
catchment        0
elevation        7
nitrate          0
richness         0
treatment       30
sample_date    116
dtype: int64


---

## Common Pitfalls

**1. Silently dropping rows with out-of-range values instead of flagging them**  
Removing rows without documentation makes results irreproducible and hides data quality problems from collaborators. Always flag suspect values first, investigate their origin, and document removals with counts and reasons. Out-of-range values are often sentinel codes (-999, 9999) or data entry errors that can be corrected.

**2. Using string comparison on unstandardised categories**  
Comparing `df['treatment'] == 'Control'` fails silently if the value is `'control'` or `' Control'`. Always strip whitespace and normalise case before any filtering or groupby on categorical columns.

**3. Treating `-999` or `0` as valid numeric values**  
Sentinel values for missing data are common in field and instrument data. A mean elevation that includes `-999` values will be wildly wrong with no error raised. Identify domain-specific sentinels early and replace them with `np.nan` before any computation.

**4. Parsing dates with `pd.to_datetime()` without `errors='coerce'`**  
Mixed date formats (`'01/07/2023'` vs `'2023-07-01'`) cause `pd.to_datetime()` to raise an exception by default. Using `errors='coerce'` converts unparseable values to `NaT` and lets you identify and fix them separately, rather than crashing on the first bad value.

**5. Not resetting the index after filtering or deduplication**  
After `drop_duplicates()` or boolean filtering, the original integer index is preserved with gaps. Downstream code that assumes a contiguous 0-based index (e.g. position-based loops or numpy conversions) will silently produce wrong results. Call `.reset_index(drop=True)` after any operation that removes rows.

---
*python_methods_library · Samantha McGarrigle · [github.com/samantha-mcgarrigle](https://github.com/samantha-mcgarrigle)*